# Homework 04 - Data Acquisition and Ingestion

## Objectives
- Download SPY price data with `yfinance`.
- Scrape the S&P 500 constituents table.
- Validate and save both datasets to `data/raw/`.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install beautifulsoup4

In [2]:
# Set paths relative to the homework folder
from pathlib import Path

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Raw data directory: {RAW}")

Working directory: D:\Desktop\NYU\FRE Bootcamp\Part3 FRE Bootcamp IV\bootcamp_project\homework\homework04
Raw data directory: D:\Desktop\NYU\FRE Bootcamp\Part3 FRE Bootcamp IV\bootcamp_project\homework\homework04\data\raw


In [3]:
import datetime as dt
import tempfile

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup

## Helpers

In [4]:
def timestamp() -> str:
    """Return a timestamp for reproducible output filenames."""
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")


def save_csv(df: pd.DataFrame, prefix: str, **metadata: str) -> Path:
    """Save a DataFrame in data/raw and return its path."""
    details = "_".join(f"{key}-{value}" for key, value in metadata.items())
    path = RAW / f"{prefix}_{details}_{timestamp()}.csv"
    df.to_csv(path, index=False)
    print(f"Saved: {path}")
    return path


def validate(df: pd.DataFrame, required_columns: list[str]) -> dict:
    """Summarize required columns, shape, missing values, and dtypes."""
    return {
        "missing_columns": [col for col in required_columns if col not in df.columns],
        "shape": df.shape,
        "na_by_column": df[required_columns].isna().sum().to_dict(),
        "dtypes": df[required_columns].dtypes.astype(str).to_dict(),
    }

## Part 1 - API Pull

Download three months of daily SPY prices through `yfinance`.

In [5]:
SYMBOL = 'SPY'
PERIOD = '3mo'
INTERVAL = '1d'

# Keep yfinance's small internal cache outside the submitted homework folder.
yfinance_cache = Path(tempfile.gettempdir()) / 'yfinance_cache'
yfinance_cache.mkdir(parents=True, exist_ok=True)
yf.set_tz_cache_location(str(yfinance_cache))

df_api = yf.download(
    SYMBOL,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    multi_level_index=False,
    progress=False,
)
if df_api.empty:
    raise RuntimeError('yfinance returned no rows. Check the network connection and try again.')

df_api = df_api.rename_axis('date').reset_index()
df_api = df_api.rename(columns={'Close': 'close'})[['date', 'close']]
df_api['date'] = pd.to_datetime(df_api['date'])
df_api['close'] = pd.to_numeric(df_api['close'], errors='coerce')

api_validation = validate(df_api, ['date', 'close'])
assert not api_validation['missing_columns']
assert df_api['date'].is_unique
assert df_api['close'].notna().all() and df_api['close'].gt(0).all()

api_validation

{'missing_columns': [],
 'shape': (64, 2),
 'na_by_column': {'date': 0, 'close': 0},
 'dtypes': {'date': 'datetime64[ns]', 'close': 'float64'}}

In [6]:
api_path = save_csv(
    df_api.sort_values("date"),
    prefix="api",
    source="yfinance",
    symbol=SYMBOL,
)
api_path

Saved: D:\Desktop\NYU\FRE Bootcamp\Part3 FRE Bootcamp IV\bootcamp_project\homework\homework04\data\raw\api_source-yfinance_symbol-SPY_20260830-111738.csv


WindowsPath('D:/Desktop/NYU/FRE Bootcamp/Part3 FRE Bootcamp IV/bootcamp_project/homework/homework04/data/raw/api_source-yfinance_symbol-SPY_20260830-111738.csv')

## Part 2 - Scrape a Public Table

Scrape the public S&P 500 constituents table from Wikipedia.

In [7]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
TABLE_ID = "constituents"
headers = {"User-Agent": "FRE5040-Homework/1.0"}

response = requests.get(SCRAPE_URL, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
table = soup.find("table", id=TABLE_ID)
if table is None:
    raise ValueError(f"Table '{TABLE_ID}' was not found.")

column_names = [cell.get_text(" ", strip=True) for cell in table.find("tr").find_all("th")]
rows = [
    [cell.get_text(" ", strip=True) for cell in row.find_all("td")]
    for row in table.find_all("tr")[1:]
]
rows = [row for row in rows if len(row) == len(column_names)]
df_scrape = pd.DataFrame(rows, columns=column_names)
df_scrape["Date added"] = pd.to_datetime(df_scrape["Date added"], errors="coerce")
df_scrape["CIK"] = df_scrape["CIK"].replace("", pd.NA)

scrape_columns = ["Symbol", "Security", "GICS Sector", "Date added", "CIK"]
scrape_validation = validate(df_scrape, scrape_columns)
assert not scrape_validation["missing_columns"]
assert not df_scrape.empty
assert df_scrape["Symbol"].is_unique
assert df_scrape[["Symbol", "Security", "GICS Sector"]].ne("").all().all()
cik_numeric = pd.to_numeric(df_scrape["CIK"], errors="coerce")
assert cik_numeric.notna().equals(df_scrape["CIK"].notna())
assert cik_numeric.dropna().ge(0).all()

scrape_validation

{'missing_columns': [],
 'shape': (503, 8),
 'na_by_column': {'Symbol': 0,
  'Security': 0,
  'GICS Sector': 0,
  'Date added': 0,
  'CIK': 0},
 'dtypes': {'Symbol': 'object',
  'Security': 'object',
  'GICS Sector': 'object',
  'Date added': 'datetime64[ns]',
  'CIK': 'object'}}

In [8]:
scrape_path = save_csv(
    df_scrape,
    prefix="scrape",
    site="wikipedia",
    table="sp500",
)
scrape_path

Saved: D:\Desktop\NYU\FRE Bootcamp\Part3 FRE Bootcamp IV\bootcamp_project\homework\homework04\data\raw\scrape_site-wikipedia_table-sp500_20260830-111739.csv


WindowsPath('D:/Desktop/NYU/FRE Bootcamp/Part3 FRE Bootcamp IV/bootcamp_project/homework/homework04/data/raw/scrape_site-wikipedia_table-sp500_20260830-111739.csv')

## Documentation

### API source
- Source: Yahoo Finance through the `yfinance` package.
- Symbol: SPY; period: 3 months; interval: daily.
- Validation: required columns, nonempty data, unique dates, and positive nonmissing closes.

### Scrape source
- Source: Wikipedia, *List of S&P 500 companies*.
- Table selector: `table#constituents`.
- Validation: required columns, nonempty and unique symbols, nonblank text fields, and missing-value counts and numeric validation for all available CIK values.

### Assumptions and risks
- Yahoo Finance or Wikipedia may change availability or column names.
- The scraped constituents are a current snapshot and may change later.
- Network failures can interrupt either download.
- `yfinance` does not require an API key.

### AI use disclosure
AI was used to help organize and check the ingestion workflow. The code, sources, and outputs were reviewed.